<a href="https://colab.research.google.com/github/gauravd12345/language_models/blob/main/seq2seq/seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [94]:
import re
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

# import nltk
# nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

dataset = "hf://datasets/PaulineSanchez/Translation_words_and_sentences_english_french/data/train-00000-of-00001-3d14582ea46e1b17.parquet"
df = pd.read_parquet(dataset)

device: cuda


In [95]:
c1, c2 = df.columns
enc = np.array(df[c1])
dec = np.array(df[c2])

for i in range(len(enc)):
  enc[i] = enc[i].lower()
  dec[i] = dec[i].lower()

for i in np.random.choice(np.arange(len(df)), 10):
  print(f"{enc[i]:<50} | {dec[i]}")

we don't want to do this.                          | nous ne voulons pas faire ceci.
do you know when they will arrive? at eleven-thirty this evening." | « savez-vous quand ils vont arriver ? » « à onze heures et demie ce soir. »
the word is on the tip of my tongue.               | j'ai le mot sur le bout de la langue.
above all, don't forget to write me.               | surtout, n'oubliez pas de m'écrire.
is the fish market open?                           | le marché aux poissons est-il ouvert ?
he is going to stay with his uncle for the weekend. | il va rester avec son oncle pour le weekend.
come on, spit it out!                              | allez, accouche !
that is what they study english for.               | c'est pour ça qu'ils étudient l'anglais.
who are you going to vote for?                     | pour qui vas-tu voter ?
if you're tired, go to bed.                        | si tu es fatigué, va te coucher.


In [96]:
""" setup for encoder dataset """
enc_words = [word_tokenize(seq) for seq in enc]
enc_vocab = set()
for seq in enc_words:
  for word in seq:
    enc_vocab.add(word)

enc_vocab.add("<sos>")
enc_vocab.add("<eos>")
enc_vocab = sorted(enc_vocab)
enc_vocab_len = len(enc_vocab)

etoi, itoe = {}, {}
for i in range(enc_vocab_len):
    etoi[enc_vocab[i]] = i
    itoe[i] = enc_vocab[i]

""" setup for decoder dataset """
dec_words = [word_tokenize(seq) for seq in dec]
dec_vocab = set()
for seq in dec_words:
  for word in seq:
    dec_vocab.add(word)

dec_vocab.add("<sos>")
dec_vocab.add("<eos>")
dec_vocab = sorted(dec_vocab)
dec_vocab_len = len(dec_vocab)

dtoi, itod = {}, {}
for i in range(dec_vocab_len):
    dtoi[dec_vocab[i]] = i
    itod[i] = dec_vocab[i]

In [97]:
print(f"english vocab size: {enc_vocab_len}")
print(np.random.choice(enc_vocab, 10))

print(f"\nfrench vocab size: {dec_vocab_len}")
print(np.random.choice(dec_vocab, 10))

english vocab size: 15132
['busier' 'unfaithful' 'inhale' 'mosque' 'environmental' 'hesitated'
 'violinist' 'world' 'faints' '229']

french vocab size: 29327
['perdes' 'sauveteurs' 'matinal' 'raisonnable' 'estimez' 'laitier'
 'perplexe' 'infirmer' 'enfoirée' 'mondes']


In [103]:
""" Hyperparameters """

embedding_dim = 300
hidden_size = 300 # context vector length
max_decoder_seq = 50

lr = 0.001
epochs = 7

enc_sos_tok = etoi["<sos>"]
enc_eos_tok = etoi["<eos>"]

dec_sos_tok = dtoi["<sos>"]
dec_eos_tok = dtoi["<eos>"]

In [99]:
E = []
for sentence in enc:
  seq = []
  tok = ["<sos>"] + word_tokenize(sentence) + ["<eos>"]
  for word in tok:
    e_i = etoi[word]
    seq.append(e_i)
  E.append(torch.tensor(seq).to(device))

D = []
for sentence in dec:
  seq = []
  tok = ["<sos>"] + word_tokenize(sentence) + ["<eos>"]
  for word in tok:
    d_i = dtoi[word]
    seq.append(d_i)
  D.append(torch.tensor(seq).to(device))

In [100]:
for i in np.random.choice(np.arange(len(E)), 10):
    english = " ".join(itoe[etok.item()] for etok in E[i])
    french  = " ".join(itod[dtok.item()] for dtok in D[i])
    print(f"[{i:2}] EN: {english:<50} | FR: {french}")

[56145] EN: <sos> never mind what he said . <eos>              | FR: <sos> peu importe ce qu'il a dit . <eos>
[25346] EN: <sos> no one is thrilled . <eos>                   | FR: <sos> personne n'est ravi . <eos>
[56552] EN: <sos> stop acting like a baby . <eos>              | FR: <sos> arrêtez de vous conduire comme un bébé ! <eos>
[154917] EN: <sos> tom is saving money for a trip to australia . <eos> | FR: <sos> tom économise de l'argent pour un voyage en australie . <eos>
[4546] EN: <sos> we might die . <eos>                         | FR: <sos> il se pourrait que nous mourions . <eos>
[69129] EN: <sos> i wanted to tell you that . <eos>            | FR: <sos> je voulais te dire ça . <eos>
[28403] EN: <sos> have a safe journey . <eos>                  | FR: <sos> fais bon voyage ! <eos>
[116413] EN: <sos> she did n't want to sell the book . <eos>    | FR: <sos> elle ne voulut pas vendre le livre . <eos>
[41860] EN: <sos> i talked to everybody . <eos>                | FR: <sos> j'ai pa

In [101]:
class Seq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_embed = nn.Embedding(enc_vocab_len, embedding_dim)
        self.dec_embed = nn.Embedding(dec_vocab_len, embedding_dim)

        """ encoder RNN """
        self.E_x = nn.Linear(embedding_dim, hidden_size)
        self.E_h = nn.Linear(hidden_size, hidden_size)
        self.E_y = nn.Linear(hidden_size, hidden_size)

        """ decoder RNN """
        self.D_x = nn.Linear(embedding_dim, hidden_size)
        self.D_h = nn.Linear(hidden_size, hidden_size)
        self.D_y = nn.Linear(hidden_size, dec_vocab_len)

    def forward(self, e, d=None):
        s = torch.zeros(hidden_size).to(device)  # initial hidden state s(0)

        w = self.enc_embed(e)
        for w_i in w:
          s = torch.tanh(self.E_x(w_i) + self.E_h(s))
          y = self.E_y(s)

        out = []
        if d is not None: # training mode(teacher forcing)
          w = self.dec_embed(d)
          for w_i in w:
            s = torch.tanh(self.D_x(w_i) + self.D_h(s))
            y = self.D_y(s)
            out.append(y)
          
        else:
          start = torch.tensor(dec_sos_tok).to(device)
          w = self.dec_embed(start)
          for _ in range(max_decoder_seq):
            s = torch.tanh(self.D_x(w) + self.D_h(s))
            y = self.D_y(s)
            out.append(y)
            pred = torch.argmax(y, dim=0).item()
          
            w = self.dec_embed(torch.tensor(pred).to(device))

        return torch.stack(out), s

seq2seq = Seq2Seq().to(device)

In [104]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(seq2seq.parameters(), lr=lr)

data_len = 2000
for epoch in range(epochs):
  total_loss = 0.0
  for e, d in zip(E[:data_len], D[:data_len]):
    optimizer.zero_grad()
    
    d_in, d_out = d[:-1], d[1:]

    out, _ = seq2seq(e[:-1], d_in)
    min_len = min(len(out), len(d_out))
      
    out = out[:min_len]
    d_out = d_out[:min_len]

    loss = criterion(out, d_out)

    total_loss += loss.item()

    loss.backward()
    optimizer.step()
  
  print(f"Epoch: {epoch+1}/{epochs} | loss: {total_loss / data_len}")

Epoch: 1/7 | loss: 2.9919311872720717
Epoch: 2/7 | loss: 2.4551768338382245
Epoch: 3/7 | loss: 2.3155900408551098
Epoch: 4/7 | loss: 2.239508883183822
Epoch: 5/7 | loss: 2.2152450639493764
Epoch: 6/7 | loss: 2.174177394753322
Epoch: 7/7 | loss: 2.140220779348165


In [ ]:
test = "how can this be?"
seq = []
tok =  ["<sos>"] + word_tokenize(test) + ["<eos>"]
for word in tok:
    e_i = etoi[word]
    seq.append(e_i)

seq = torch.tensor(seq).to(device)
pred, _ = seq2seq(seq)
for prob in pred:
    word = itod[torch.argmax(prob).item()]
    if word == "<eos>":
        break

    print(word, end="")

moiaussi.